In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from esda import Moran
import geopandas as gpd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from libpysal.weights import KNN
from spreg import OLS, GM_Error_Het
from statsmodels.stats.outliers_influence import variance_inflation_factor

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
property_old = gpd.read_file('output/property.gpkg')
property_reach_canopies = gpd.read_file('output/property_isodistance_canopies.gpkg')

In [3]:
property_full = property_old.merge(
    property_reach_canopies[[
      'property_id',
      'canopy_25m',
      'canopy_75m',
      'canopy_50m',
      'canopy_100m',
      'canopy_150m',
      'canopy_200m',
      'canopy_250m',
      'canopy_300m',
      'canopy_350m',
      'canopy_400m'
    ]],
    on='property_id',
    how='left'
)

In [4]:
property_full.columns

Index(['GrossSalePrice', 'AgeAtSale', 'LandArea', 'TotalFloorArea',
       'water_DIST', 'bus_DIST', 'Census_Pop', 'RnkIMDNoEm', 'RnkIMDNoIn',
       'RnkIMDNoCr', 'RnkIMDNoHo', 'RnkIMDNoHe', 'RnkIMDNoEd', 'RnkIMDNoAc',
       'DECILE_high', 'DECILE_prim', 'Median_Income', 'CBD_DIST',
       'cycleways_DIST', 'cycle_DENS', 'year_2018', 'year_2019', 'canopy_0_20',
       'canopy_20_50', 'canopy_50_100', 'canopy_100_200', 'residuals',
       'property_id', 'geometry', 'canopy_25m', 'canopy_75m', 'canopy_50m',
       'canopy_100m', 'canopy_150m', 'canopy_200m', 'canopy_250m',
       'canopy_300m', 'canopy_350m', 'canopy_400m'],
      dtype='object')

In [5]:
property = property_full.drop(columns=['property_id', 'canopy_0_20', 'canopy_20_50', 'canopy_50_100', 'canopy_100_200', 'residuals'])

In [6]:
property['canopy_0_25m'] = property['canopy_25m']
property['canopy_25_50m'] = property['canopy_50m'] - property['canopy_25m']
property['canopy_25_75m'] = property['canopy_75m'] - property['canopy_25m']
property['canopy_25_100m'] = property['canopy_100m'] - property['canopy_25m']
property['canopy_50_75m'] = property['canopy_75m'] - property['canopy_50m']
property['canopy_50_100m'] = property['canopy_100m'] - property['canopy_50m']
property['canopy_50_150m'] = property['canopy_150m'] - property['canopy_50m']
property['canopy_50_200m'] = property['canopy_200m'] - property['canopy_50m']
property['canopy_50_250m'] = property['canopy_250m'] - property['canopy_50m']
property['canopy_50_300m'] = property['canopy_300m'] - property['canopy_50m']
property['canopy_50_350m'] = property['canopy_350m'] - property['canopy_50m']
property['canopy_50_400m'] = property['canopy_400m'] - property['canopy_50m']
property['canopy_75_100m'] = property['canopy_100m'] - property['canopy_75m']
property['canopy_75_150m'] = property['canopy_150m'] - property['canopy_75m']
property['canopy_75_200m'] = property['canopy_200m'] - property['canopy_75m']
property['canopy_75_250m'] = property['canopy_250m'] - property['canopy_75m']
property['canopy_75_300m'] = property['canopy_300m'] - property['canopy_75m']
property['canopy_75_350m'] = property['canopy_350m'] - property['canopy_75m']
property['canopy_75_400m'] = property['canopy_400m'] - property['canopy_75m']
property['canopy_100_150m'] = property['canopy_150m'] - property['canopy_100m']
property['canopy_150_200m'] = property['canopy_200m'] - property['canopy_150m']
property['canopy_200_250m'] = property['canopy_250m'] - property['canopy_200m']
property['canopy_250_300m'] = property['canopy_300m'] - property['canopy_250m']
property['canopy_300_350m'] = property['canopy_350m'] - property['canopy_300m']
property['canopy_350_400m'] = property['canopy_400m'] - property['canopy_350m']

In [7]:
property.head()

,GrossSalePrice,AgeAtSale,LandArea,TotalFloorArea,water_DIST,bus_DIST,Census_Pop,RnkIMDNoEm,RnkIMDNoIn,RnkIMDNoCr,...,canopy_75_250m,canopy_75_300m,canopy_75_350m,canopy_75_400m,canopy_100_150m,canopy_150_200m,canopy_200_250m,canopy_250_300m,canopy_300_350m,canopy_350_400m
0,610000,24,685.0,211.0,189.601703,216.798910,930.0,1811.0,849.0,1231.0,...,478.832883,611.215459,861.067219,1171.755968,12.057335,114.782206,230.481210,132.382576,249.851760,310.688749
1,477500,42,609.0,120.0,139.581200,154.824709,930.0,1811.0,849.0,1231.0,...,812.855431,1106.808787,1564.210915,2093.473450,253.216186,131.881281,350.951746,293.953356,457.402128,529.262535
2,655000,22,704.0,245.0,201.942495,192.941029,930.0,1811.0,849.0,1231.0,...,97.969902,343.202418,525.795101,702.754327,28.485955,14.573724,46.982507,245.232516,182.592683,176.959226
3,500000,44,869.0,131.0,157.854014,179.645498,930.0,1811.0,849.0,1231.0,...,876.668880,1103.262590,1594.619219,2279.765973,188.617149,157.877068,397.685651,226.593710,491.356629,685.146753
4,880000,22,1149.0,279.0,199.693219,216.730600,930.0,1811.0,849.0,1231.0,...,223.199317,470.303137,604.635344,894.414005,6.493093,28.272726,162.918491,247.103820,134.332207,289.778661


In [8]:
property['log_price'] = np.log(property['GrossSalePrice'])


In [9]:
w = KNN.from_dataframe(property, k=8)
w.transform = 'R'

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


In [10]:
y = property['log_price'].values.reshape(-1, 1)

In [28]:
non_x_cols = [
  # unneccessary columns
  'GrossSalePrice', 'geometry', 'log_price',
  # removed columns 
  # "AgeAtSale",  # include
  # "Census_Pop",
  # "RnkIMDNoEm",
  # "RnkIMDNoIn",
  # "RnkIMDNoCr",
  # "RnkIMDNoHo",
  # "RnkIMDNoHe",
  # "RnkIMDNoEd",
  # "RnkIMDNoAc",
  # "DECILE_high",
  # "Median_Income", # include
  "DECILE_prime",
  'TotalFloorArea',
  'LandArea',
  'water_DIST',
  'bus_DIST',
  'DECILE_prim',
  'CBD_DIST',
  'cycleways_DIST',
  'cycle_DENS',
  'year_2018',
  'year_2019',
  'canopy_25m',
  'canopy_50m',
  'canopy_75m',
  'canopy_100m',
  'canopy_150m',
  'canopy_200m',
  'canopy_250m',
  'canopy_300m',
  'canopy_350m',
  'canopy_400m',
  'canopy_0_25m',
  'canopy_25_50m',
  'canopy_25_75m',
  'canopy_25_100m',
  'canopy_50_75m',
  'canopy_50_100m', # significant
  'canopy_50_150m', 
  'canopy_50_200m',
  'canopy_50_250m',
  'canopy_50_300m',
  'canopy_50_350m',
  'canopy_50_400m',
  # 'canopy_75_100m', # 0.001 significant
  'canopy_75_150m',
  'canopy_75_200m',
  'canopy_75_250m',
  'canopy_75_300m',
  'canopy_75_350m',
  'canopy_75_400m',
  'canopy_100_150m',
  'canopy_150_200m',
  'canopy_200_250m',
  'canopy_250_300m',
  'canopy_300_350m',
  'canopy_350_400m'
]

x_cols = [col for col in property.columns if col not in non_x_cols]

print(x_cols)

X = property.loc[:, x_cols]

X = X.values

['AgeAtSale', 'Census_Pop', 'RnkIMDNoEm', 'RnkIMDNoIn', 'RnkIMDNoCr', 'RnkIMDNoHo', 'RnkIMDNoHe', 'RnkIMDNoEd', 'RnkIMDNoAc', 'DECILE_high', 'Median_Income', 'canopy_75_100m']


In [29]:
X = property.loc[:, x_cols]
X = X.values
sem = GM_Error_Het(y, X, w, name_y = 'log_price', name_x = x_cols)
print(sem.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES (HET)
------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :   log_price                Number of Observations:       12317
Mean dependent var  :     13.1505                Number of Variables   :          13
S.D. dependent var  :      0.3355                Degrees of Freedom    :       12304
Pseudo R-squared    :      0.4660
N. of iterations    :           1                Step1c computed       :          No

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        13.03899         0.04856       268.50515         0.00000
           AgeAtSale        -0.00281         0.00010    